## Finding Cutoff Emipirically

In [4]:
# adding librarieas 
import numpy 
import time
import sys
import pandas as pd

In [ ]:
#adding necessary functions 

def add_subtract_matrix(A, B, operation=1):
    """ Adds or subtracts two matrices using NumPy """
    return np.add(A, operation * B)

def strassen(A, B, n = 1):
    dim = A.shape[0]
    
    #if the dimension is 1 or is under the cutoff then just do naive implementation 
    if dim == 1 or dim <= n:
        return matrix_mult(A, B)
    
    newDim = dim // 2
    
    #split up matrices 
    a, b, c, d = A[:newDim, :newDim], A[:newDim, newDim:], A[newDim:, :newDim], A[newDim:, newDim:]
    e, f, g, h = B[:newDim, :newDim], B[:newDim, newDim:], B[newDim:, :newDim], B[newDim:, newDim:]

    #do 7 multiplications 
    p1 = strassen(a, add_subtract_matrix(f, h, -1))
    p2 = strassen(add_subtract_matrix(a, b), h)
    p3 = strassen(add_subtract_matrix(c, d), e)
    p4 = strassen(d, add_subtract_matrix(g, e, -1))
    p5 = strassen(add_subtract_matrix(a, d), add_subtract_matrix(e, h))
    p6 = strassen(add_subtract_matrix(b, d, -1), add_subtract_matrix(g, h))
    p7 = strassen(add_subtract_matrix(c, a, -1), add_subtract_matrix(e, f))
    
    #AE + BG = -p2 + p4 + p5 + p6
    C11 = add_subtract_matrix(add_subtract_matrix(p4, p2, -1), add_subtract_matrix(p6, p5))

    #AF + BH = p1 + p2
    C12 = add_subtract_matrix(p1, p2)

    #CE + DG = p3 + p4
    C21 = add_subtract_matrix(p3, p4)

    #CF + DH = p1 - p3 + p5 + p7
    C22 = add_subtract_matrix(add_subtract_matrix(p1, p3, -1), add_subtract_matrix(p5, p7))
    
    #construct result matrix using numpy
    C = np.block([[C11, C12], [C21, C22]])

    
    return C

def next_power_of_2(n):
    """ Returns the next power of 2 greater than or equal to n """
    return 2**int(np.ceil(np.log2(n)))

def strassen_pad(A, B, n):


    dim = A.shape[0]
    newDim = next_power_of_2(dim)
  
    A_padded = np.pad(A, ((0, newDim - dim), (0, newDim - dim)), mode='constant')
    B_padded = np.pad(B, ((0, newDim - dim), (0, newDim - dim)), mode='constant')
   
    C_padded = strassen(A_padded, B_padded, n)
    
    return C_padded[:dim, :dim]
    
def matrix_mult(A, B):
    """Performs naive matrix multiplication of two square matrices A and B."""
    n = len(A)
    product = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            for k in range(n):
                product[i, j] += A[i, k] * B[k, j]
    return product

def generate_random_matrix(n):
    """Generates an n x n matrix with entries randomly selected from value_set."""
    return np.random.int(0, 2, size=(n, n))

def gendata(d): 

    #generate random matrices A and B 
    A, B = generate_random_matrix(d), generate_random_matrix(d)

    #find the time it takes to run naive multiplication 
    start = time.time()
    matrix_mult(A, B)
    end = time.time()
    naive_time = end - start

    #find the time it takes to run strassen for every possible cut off
    strassen_times = []
    for cutoff in range(1, d+1): 
        start = time.time()
        strassen_pad(A, B, cutoff)
        end = time.time()
        strassen_times.append(end - start)

    return naive_time, strassen_times


def find_cutoff(d): 
    """Okay this returns a dataframe with dimensions and the time it took for naive plus strassen 
    variations to actually do the multiplication"""

    columns =['dimension', 'naive_time'] + [f'strassen_{i}' for i in range(1, d+1)]
    df = pd.DataFrame(columns = columns)

    for dim in range(1, d + 1): 

        #okay this creats data for dimensions 1 to d
        naive_time, strassen_times = gendata(dim)

        row = {'dimension': dim, 'naive_time': naive_time}
        
        # ok add the strassen variation cuttoff times 
        for i in range(1, dim+1):
            row[f'strassen_{i}'] = strassen_times[i-1]  # Assuming strassen_times has at least D values

        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

    return df 




### Now going to collect some data 

In [6]:
df = find_cutoff(20)
df.head()

TypeError: generate_random_matrix() missing 1 required positional argument: 'value_set'